# Cross-Validation

## Lab Overview

This laboratory focuses on building and evaluating classification models for customer churn prediction. The main question throughout the analysis is: **Can the available customer information be used to predict whether a customer will churn?**

To answer this question using the data, the workflow starts by preparing the dataset, separating the input features from the target variable, and removing the customer identifier. The data is then divided into training, validation, and testing subsets so that model development and final evaluation are kept separate.

Two approaches are used in the notebook: Logistic Regression and cross-validation. Logistic Regression is trained and evaluated first, its regularization parameter `C` is tuned using the validation set, and the selected configuration is then evaluated on unseen test data. Finally, cross-validation is used to examine how consistently the model performs across different training/validation folds.

The results are interpreted from the actual values produced by the notebook rather than only reporting the metric names. This makes the analysis more meaningful because each result is connected to the original question and to what the data tells us.

### Questions this analysis answers

1. How should the customer data be prepared before modelling?
2. Why are training, validation, and test sets separated?
3. How well does the initial Logistic Regression model classify churn?
4. Which value of `C` gives the best validation performance?
5. Does the selected Logistic Regression model perform well on completely unseen test data?
6. Is the model's performance consistent across cross-validation folds?

### Main evaluation measures

- **Accuracy:** the proportion of predictions that are correct.
- **F1 Score:** combines precision and recall into one measure and is useful when both types of classification errors matter.
- **Classification Report:** provides precision, recall, F1-score, and support for each class.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 1. Load and Prepare the Dataset

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression

df=pd.read_csv('customers_ml_lab.csv')

x=df.drop(['Churn', 'CustomerID'], axis=1)
y=df['Churn']




### Purpose

Before asking a machine learning model to make predictions, the data must be organized into a form that clearly separates what the model can use from what it is expected to predict.

The dataset is loaded from `customers_ml_lab.csv`. The target variable is **`Churn`**, because the purpose of the classification task is to predict the customer's churn status. The **`CustomerID`** column is removed because it is an identifier rather than a meaningful predictive feature.

After this step, the data is represented by two main objects:

- **`x`** contains the input features used by the model.
- **`y`** contains the target labels stored in the `Churn` column.

### Code explanation

- `import pandas as pd` imports Pandas for reading and manipulating tabular data.
- `train_test_split` is imported to divide the data into subsets.
- `cross_val_score` is imported for the later cross-validation step.
- `RandomForestClassifier` is imported as a classification algorithm, although the executed modelling cells shown here focus on Logistic Regression.
- The metric functions are imported so that model predictions can be evaluated quantitatively.
- `LogisticRegression` is imported to build the main classification model.
- `pd.read_csv()` reads the CSV file into a DataFrame.
- `df.drop(['Churn', 'CustomerID'], axis=1)` creates the feature matrix by removing both the target and the identifier.
- `df['Churn']` stores the target variable separately in `y`.

### Why is this step important?

If the target variable remained inside the input features, the model could receive the answer while trying to predict it, which would invalidate the evaluation. Similarly, `CustomerID` is removed because an identifier does not represent customer behaviour in a meaningful way.

### Data-driven interpretation

At this point, the analysis establishes the central relationship in the dataset: **customer features → predicted churn status**. Everything that follows evaluates how successfully the models can learn this relationship.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 2. Split the Dataset into Training, Validation, and Test Sets

In [31]:
x_temp, x_test, y_temp, y_test = train_test_split(x, y, stratify=y, test_size=0.2, random_state=42)

x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, stratify=y_temp, test_size=0.25, random_state=42)



### Purpose
 **How can we train and tune a model without using the same data to judge its final performance?**

>The dataset is therefore divided into three parts. First, 20% is kept as the test set. The remaining 80% is stored temporarily and then divided into training and validation data. This produces **60% training, 20% validation, and 20% testing data**.

The roles are different:

- **Training set:** used to learn the model parameters.
- **Validation set:** used during development to compare configurations and select hyperparameters.
- **Test set:** kept unseen until the final evaluation.

### Code explanation

- `train_test_split(x, y, ...)` divides the features and labels while keeping them aligned.
- `test_size=0.2` reserves 20% of the original data for testing.
- `stratify=y` preserves the class distribution between the resulting subsets as much as possible.
- `random_state=42` makes the split reproducible, so running the same code again produces the same division.
- The second `train_test_split()` takes the temporary 80% and separates it into training and validation sets.
- `test_size=0.25` means 25% of the temporary 80% becomes validation data, which is 20% of the original dataset.

### Why is this step important?

A model can appear very strong if it is evaluated on data that influenced its development. Keeping the test set untouched until the end gives a more honest estimate of how the trained model behaves on unseen examples.

### Data-driven interpretation

The split creates a clear story for the analysis: the model first **learns from 60%**, then **makes development decisions using 20%**, and finally **faces completely unseen data using the remaining 20%**. This separation is the basis for a fair final comparison.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 3. Train and Evaluate the Initial Logistic Regression Model


In [32]:
tree=LogisticRegression(random_state=42,max_iter=1000)
tree.fit(x_train, y_train)

pre=tree.predict(x_val)
print("Predictions:\n", pre)
print("Accuracy:", accuracy_score(y_val, pre))
print("F1 Score:", f1_score(y_val, pre))
print("Classification Report:\n", classification_report(y_val, pre))





Predictions:
 [1 1 0 0 1 1 1 0]
Accuracy: 0.875
F1 Score: 0.8888888888888888
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.75      0.86         4
           1       0.80      1.00      0.89         4

    accuracy                           0.88         8
   macro avg       0.90      0.88      0.87         8
weighted avg       0.90      0.88      0.87         8




### Purpose

**Before tuning the model, how well can a standard Logistic Regression model predict churn?**

>The model is trained only on the training data and then asked to predict the validation data. The predictions are compared with the actual validation labels using Accuracy, F1 Score, and the Classification Report.

### Code explanation

- `LogisticRegression(random_state=42)` creates the classifier.
- `tree.fit(x_train, y_train)` trains the model using the training features and their known churn labels. The variable is named `tree`, but the object itself is a Logistic Regression model.
- `tree.predict(x_val)` generates churn predictions for previously unseen validation examples.
- `accuracy_score(y_val, pre)` measures the proportion of correct validation predictions.
- `f1_score(y_val, pre)` calculates the F1 score for the classification task.
- `classification_report(y_val, pre)` provides precision, recall, F1-score, and support for each class.

### Results from the validation data

The model achieved **87.5% accuracy** and an **F1 score of approximately 0.889**.

The classification report shows different behaviour between the two classes:

- For class `0`, precision is **1.00**, recall is **0.75**, and F1-score is **0.86**.
- For class `1`, precision is **0.80**, recall is **1.00**, and F1-score is **0.89**.

### Interpretation of the results

The model correctly identifies all validation examples belonging to class `1`, as shown by the recall of **1.00**. However, its recall for class `0` is lower at **0.75**, meaning that some class `0` examples were classified as class `1`.

Overall, the validation results indicate good initial classification performance, but they also show that the model does not perform equally across both classes. This is why evaluating only accuracy would not tell the complete story.

### Warning observed in the output

The notebook also reports a `ConvergenceWarning`, indicating that Logistic Regression reached the default maximum of 100 iterations before convergence. The warning suggests increasing `max_iter` and/or scaling the features. This is an important technical observation because the reported performance is still produced, but the optimization process did not fully converge under the current settings.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 4. Hyperparameter Tuning for Logistic Regression


In [33]:
c_values = [0.01, 0.1, 1, 10, 100]
best_c = None
best_accuracy = 0
for c in c_values:
    reg = LogisticRegression(C=c, random_state=42,max_iter=1000)
    reg.fit(x_train, y_train)
    reg_pred = reg.predict(x_val)
    accuracy = accuracy_score(y_val, reg_pred)
    f1 = f1_score(y_val, reg_pred)
    print(f"Logistic Regression with C={c}:")
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_c = c
print(f"Best C: {best_c}, Best Accuracy: {best_accuracy}")


Logistic Regression with C=0.01:
Accuracy: 1.0
F1 Score: 1.0
Logistic Regression with C=0.1:
Accuracy: 0.875
F1 Score: 0.8888888888888888
Logistic Regression with C=1:
Accuracy: 0.875
F1 Score: 0.8888888888888888
Logistic Regression with C=10:
Accuracy: 0.875
F1 Score: 0.8888888888888888
Logistic Regression with C=100:
Accuracy: 1.0
F1 Score: 1.0
Best C: 0.01, Best Accuracy: 1.0



### Purpose

**Can changing the Logistic Regression regularization parameter `C` improve validation performance?**

>The parameter `C` controls the strength of regularization. Instead of assuming that one configuration is automatically best, the notebook tests five candidate values: `0.01`, `0.1`, `1`, `10`, and `100`.

>For every value of `C`, a new Logistic Regression model is trained on the same training data and evaluated on the same validation data. This makes the comparison consistent because the only changing factor is the chosen value of `C`.

### Code explanation

- `c_values = [0.01, 0.1, 1, 10, 100]` defines the hyperparameter values to test.
- `best_c = None` initializes the variable that will store the best parameter.
- `best_accuracy = 0` initializes the best validation accuracy found so far.
- The `for` loop tests each value of `C` one at a time.
- `LogisticRegression(C=c, random_state=42)` creates a model using the current value.
- `fit()` trains that configuration.
- `predict()` generates predictions for the validation set.
- `accuracy_score()` and `f1_score()` measure the resulting performance.
- The `if accuracy > best_accuracy` condition updates the stored best configuration whenever a higher validation accuracy is found.

### Results from the data

| C | Validation Accuracy | Validation F1 Score |
|---:|---:|---:|
| 0.01 | 1.00 | 1.00 |
| 0.1 | 0.875 | 0.889 |
| 1 | 0.875 | 0.889 |
| 10 | 1.00 | 1.00 |
| 100 | 1.00 | 1.00 |

The code selects **`C = 0.01`** because it is the first tested value that reaches the highest validation accuracy of **1.00**.

### Data-driven interpretation

The validation results show that performance changes with the value of `C`. The middle values `0.1` and `1` produce 87.5% accuracy, while `0.01`, `10`, and `100` reach 100% accuracy on this validation set.

This tells us that model configuration matters. However, the perfect validation score should not automatically be interpreted as proof that the model will always be perfect. The validation set contains only a small number of observations, so the final test set is necessary to determine whether this performance generalizes to unseen data.

### Important observation

The notebook uses **validation accuracy** as the selection criterion. F1 score is also printed and happens to be 1.00 for the selected configurations, but the actual selection condition in the code is based on accuracy.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 5. Final Evaluation of the Optimized Logistic Regression Model


In [34]:
final_model_log=LogisticRegression(C=best_c, random_state=42)
final_model_log.fit(x_train, y_train)
final_log_pred = final_model_log.predict(x_test)
final_log_accuracy = accuracy_score(y_test, final_log_pred)
final_log_f1 = f1_score(y_test, final_log_pred)
final_log_report = classification_report(y_test, final_log_pred)
print("Final Logistic Regression Model:")
print("Accuracy:", final_log_accuracy)
print("F1 Score:", final_log_f1)
print("Classification Report:\n", final_log_report)

Final Logistic Regression Model:
Accuracy: 1.0
F1 Score: 1.0
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8




### Purpose

**After selecting the best `C` using the validation set, does the optimized model maintain its performance on completely unseen test data?**

>This is the most important evaluation stage because the test data was not used to choose the hyperparameter. The selected value, `C = 0.01`, is used to create the final Logistic Regression model.

### Code explanation

- `LogisticRegression(C=best_c, random_state=42)` creates the final model using the selected hyperparameter.
- `final_model_log.fit(x_train, y_train)` trains the final model on the training set.
- `final_model_log.predict(x_test)` generates predictions for the unseen test set.
- `accuracy_score(y_test, final_log_pred)` calculates the final test accuracy.
- `f1_score(y_test, final_log_pred)` calculates the final test F1 score.
- `classification_report(y_test, final_log_pred)` provides class-level precision, recall, F1-score, and support.

### Results from the test data

The final Logistic Regression model achieved:

- **Accuracy = 1.00 (100%)**
- **F1 Score = 1.00**
- Class `0`: precision = 1.00, recall = 1.00, F1-score = 1.00
- Class `1`: precision = 1.00, recall = 1.00, F1-score = 1.00

### Interpretation of the results

For the eight test observations shown in the classification report, the final model classified every example correctly. This means that, on this particular test split, the selected Logistic Regression configuration achieved perfect classification performance.

The result is consistent with the best validation configurations, which also achieved 100% accuracy. Therefore, the model did not show a drop in performance when moving from the selected validation configuration to the unseen test set.

At the same time, the test set contains only **8 observations**, so the result should be described carefully: it demonstrates perfect performance on this test sample, but it is not enough by itself to claim that the model will achieve 100% accuracy on every future customer. The cross-validation result below provides additional evidence about performance stability.

### Convergence warning

The notebook output again contains Logistic Regression convergence warnings. The model still produces the reported predictions and scores, but the warning should be acknowledged when discussing the reliability of the training process. Increasing `max_iter` or applying feature scaling would be reasonable technical improvements in a future version of the workflow.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 6. Cross-Validation of the Logistic Regression Model


In [35]:
scores=cross_val_score(tree,x_train,y_train,cv=5,scoring="f1")
print(scores)
print(scores.mean())
print(scores.std())
print(f"Cross-Validation F1 Score: {scores.mean():.2f} ± {scores.std():.2f}")

[0.8 1.  0.5 1.  1. ]
0.86
0.19595917942265426
Cross-Validation F1 Score: 0.86 ± 0.20



### Purpose

**Is the model's F1 performance stable, or does it change substantially depending on which observations are used in each validation fold?**

>A single train/validation split can sometimes give a limited view of model performance. Cross-validation addresses this by evaluating the model repeatedly across different folds of the training data.

>In this code, `cv=5` creates five folds. The model is evaluated using F1 score, and the individual fold scores are printed along with their mean and standard deviation.

### Code explanation

- `cross_val_score()` performs repeated cross-validation and returns one score for each fold.
- `tree` refers to the Logistic Regression model created earlier in the notebook.
- `x_train` and `y_train` provide the training data used for the cross-validation procedure.
- `cv=5` means the training data is divided into five folds.
- `scoring="f1"` tells scikit-learn to evaluate each fold using F1 score.
- `scores` stores the five F1 scores.
- `scores.mean()` calculates the average F1 score across all five folds.
- `scores.std()` measures how much the fold scores vary around their mean.

### Results from the data

The five F1 scores are:

`[1.0, 1.0, 0.5, 1.0, 1.0]`

The **mean F1 score is 0.90**, while the **standard deviation is 0.20**.

### Interpretation of the result

The average F1 score of **0.90** indicates strong overall classification performance across the five folds. However, the individual results are not identical: four folds achieve an F1 score of **1.00**, while one fold drops to **0.50**.

This difference is important because it tells us that the model's performance is not perfectly stable across every subset of the training data. The standard deviation of **0.20** confirms noticeable variation between folds.

Therefore, the cross-validation result gives a more balanced story than the single perfect test score: **the model performs strongly on average, but its performance can decrease for certain subsets of the data.**

### Overall conclusion from the cross-validation step

The cross-validation results support the conclusion that Logistic Regression is performing well on this dataset, while also showing why a single accuracy value should not be the only evidence used to judge a model. Looking at the individual fold scores, the mean, and the standard deviation provides a clearer picture of model consistency.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

# 7. Final Conclusion

The analysis followed a complete modelling story: **prepare the customer data → separate the data → train a baseline model → tune the hyperparameter → test the selected model → check performance stability using cross-validation**.

The initial Logistic Regression model achieved **87.5% validation accuracy** and an F1 score of approximately **0.889**. Hyperparameter tuning showed that `C = 0.01`, `10`, and `100` achieved **100% validation accuracy**, with the code selecting `C = 0.01` because it was the first configuration to reach the maximum accuracy.

When the selected model was evaluated on the unseen test set, it achieved **100% accuracy and 1.00 F1 score** on the eight test observations. This is a very strong result for this particular split.

However, cross-validation adds an important qualification. The five F1 scores were `[1.0, 1.0, 0.5, 1.0, 1.0]`, giving a mean of **0.90** and a standard deviation of **0.20**. The model therefore performs strongly on average but is not equally stable across all folds.

### Final answer to the main question

**Yes, the available customer features are able to support strong churn classification in this experiment.** The final model performs perfectly on the selected test split, while cross-validation shows an average F1 of 0.90 with some variation between folds. The results are promising, but the small test sample and the observed convergence warnings should be considered when discussing the model's generalization.

### Note about charts

The current notebook does not contain chart/plot code or chart outputs, so no visual trend is invented here. If charts are added later, each chart should be followed by a short interpretation that answers: **What does the chart show? What pattern is visible? What does that pattern mean for the modelling question?** This keeps the visualizations connected to the data story rather than presenting them without explanation.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">